# MEGAN 模型架构与推理展示

**MEGAN** (Molecule Edit Graph Attention Network) 的模型部分包含三个核心组件：

```
                     ┌─────────────┐
  分子图 ────────────►│  GAT 编码器  │──── 节点嵌入
  (节点 + 邻接矩阵)   │ (Encoder)   │
                     └─────────────┘
                            │
                            ▼
                     ┌─────────────┐
                     │  GAT 解码器  │──── 动作概率
                     │ (Decoder)   │
                     └─────────────┘
                       /         \
                      ▼           ▼
              原子动作头       键动作头
           (n_nodes × n_atom_actions)  (n_nodes × n_nodes × n_bond_actions)
                      \           /
                       ▼         ▼
                    Softmax → 选择动作
                         │
                         ▼
                    执行动作，更新分子图
                    重复直到 Stop
```

## 本 Notebook 内容

| 章节 | 内容 |
|------|------|
| Step 1 | 环境与导入 |
| Step 2 | 多头图注意力层 (MultiHeadGraphConvLayer) |
| Step 3 | GAT 编码器 (MeganEncoder) |
| Step 4 | GAT 解码器与动作预测头 (MeganDecoder) |
| Step 5 | MEGAN 完整模型 (Megan) |
| Step 6 | 自回归推理与 Beam Search |

**参考论文**: Mikołaj Sacha et al., "Molecule Edit Graph Attention Network: Modeling Chemical Reactions as Sequences of Graph Edits", *JCIM*, 2021

In [6]:
# ============================================================
# Step 1: 环境与导入
# ============================================================
import os
import sys
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable

# 路径设置（与前两个 Notebook 一致）
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..', '..'))

MEGAN_SOURCE = os.path.join(PROJECT_ROOT, 'source_repos', 'megan')
MEGAN_ENV = os.path.join(PROJECT_ROOT, 'envs', 'megan_tutorial_envs')
DATA_DIR = os.path.join(NOTEBOOK_DIR, 'data')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print(f"PyTorch 版本: {torch.__version__}")
print(f"设备: {device}")
print(f"MEGAN 源码: {MEGAN_SOURCE}")

# 验证 RDKit
from rdkit import Chem
print(f"RDKit 版本: {Chem.rdBase.rdkitVersion}")
print("\n✅ 环境就绪")

PyTorch 版本: 2.5.1+cu121
设备: cuda:0
MEGAN 源码: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/megan
RDKit 版本: 2025.03.5

✅ 环境就绪


## Step 2: 多头图注意力层 (MultiHeadGraphConvLayer)

这是 MEGAN 的基础构建块。与标准 GAT 的区别在于：**MEGAN 在注意力计算中同时考虑了边特征（bond embedding）**。

### 注意力计算

标准 GAT:
$$\alpha_{ij} = \text{softmax}_j \left( \text{LeakyReLU}(\mathbf{a}^T [\mathbf{W}\mathbf{h}_i \| \mathbf{W}\mathbf{h}_j]) \right)$$

MEGAN 改进（加入边特征）:
$$\alpha_{ij} = \text{softmax}_j \left( \mathbf{W}_{att} [\mathbf{W}_q \mathbf{h}_i \| \mathbf{W}_q \mathbf{h}_j \| \mathbf{e}_{ij}] \right)$$

其中 $\mathbf{e}_{ij}$ 是边（化学键）的嵌入向量。

### 多头注意力
- 默认 8 个注意力头 (`att_heads=8`)
- 每个头独立计算注意力和聚合
- 最终拼接所有头的输出：$\mathbf{h}'_i = \|_{k=1}^{K} \mathbf{W}_k (\sum_j \alpha_{ij}^k \mathbf{h}_j)$

In [7]:
# ============================================================
# Step 2: 实现 MultiHeadGraphConvLayer
# ============================================================

class MultiHeadGraphConvLayer(nn.Module):
    """
    多头图注意力卷积层。
    
    与标准 GAT 的关键区别：
    - 注意力计算中加入了边特征 (bond embedding)
    - 使用 softmax + mask 处理不规则图
    """
    def __init__(self, bond_dim, input_dim, output_dim, residual=False,
                 att_heads=8, att_dim=64):
        """
        参数:
            bond_dim: 边嵌入维度
            input_dim: 输入节点特征维度
            output_dim: 输出节点特征维度（必须是 att_heads 的倍数）
            att_heads: 注意力头数
            att_dim: 注意力中间维度
        """
        super().__init__()
        self.n_att = att_heads
        self.att_dim = att_dim
        
        # 注意力网络：将节点对 + 边特征映射到注意力分数
        self.atoms_att = nn.Linear(input_dim, att_dim)
        self.final_att = nn.Linear(att_dim * 2 + bond_dim, att_heads)
        
        # 每个头一个独立的线性变换
        assert output_dim % att_heads == 0, f"output_dim({output_dim}) 必须是 att_heads({att_heads}) 的倍数"
        self.conv_layers = nn.ModuleList([
            nn.Linear(input_dim, output_dim // att_heads)
            for _ in range(att_heads)
        ])
        self.residual = residual
    
    def forward(self, x, adj, mask, soft_mask, apply_activation=True):
        """
        参数:
            x: (batch, n_nodes, input_dim) 节点特征
            adj: (batch, n_nodes, n_nodes, bond_dim) 边嵌入
            mask: (batch, n_nodes, n_nodes) 邻接掩码 (0/1)
            soft_mask: (batch, n_nodes, n_nodes) softmax 掩码 (-1e9/0)
            apply_activation: 是否应用 ReLU
        """
        # 1. 计算注意力: 对每对节点 (i,j)，拼接 h_i, h_j, e_ij
        x_att = torch.relu(self.atoms_att(x))                    # (B, N, att_dim)
        x_att_shape = adj.shape[:-1] + (x_att.shape[-1],)
        x_rows = x_att.unsqueeze(1).expand(x_att_shape)          # (B, N, N, att_dim) - 行节点
        x_cols = x_att.unsqueeze(2).expand(x_att_shape)          # (B, N, N, att_dim) - 列节点
        
        x_att = torch.cat([x_rows, x_cols, adj], dim=-1)         # (B, N, N, 2*att_dim+bond_dim)
        x_att = self.final_att(x_att)                            # (B, N, N, att_heads)
        
        # 2. 多头聚合
        head_outs = []
        for i, conv in enumerate(self.conv_layers):
            att = x_att[:, :, :, i]                              # (B, N, N) 第 i 个头的注意力
            att = torch.softmax(att + soft_mask, dim=-1) * mask  # 归一化 + 掩码
            out = torch.bmm(att, x)                              # (B, N, input_dim) 聚合
            out = conv(out)                                      # (B, N, output_dim/heads) 变换
            head_outs.append(out)
        
        out = torch.cat(head_outs, dim=-1)                       # (B, N, output_dim)
        
        if self.residual:
            out = x + out
        if apply_activation:
            out = torch.relu(out)
        
        return out


# === 演示 ===
print("=" * 60)
print("Step 2: MultiHeadGraphConvLayer 演示")
print("=" * 60)

# 模拟一个 3 节点的分子图 (含超节点)
batch_size = 1
n_nodes = 3
input_dim = 64
bond_dim = 8
output_dim = 64

gat_layer = MultiHeadGraphConvLayer(
    bond_dim=bond_dim, input_dim=input_dim, output_dim=output_dim,
    att_heads=8, att_dim=32
)

# 模拟输入
x = torch.randn(batch_size, n_nodes, input_dim)
adj = torch.randn(batch_size, n_nodes, n_nodes, bond_dim)

# 掩码：表示哪些节点对之间存在"可能的边"
mask = torch.ones(batch_size, n_nodes, n_nodes)
soft_mask = torch.zeros(batch_size, n_nodes, n_nodes)

out = gat_layer(x, adj, mask, soft_mask)
print(f"\n输入: x.shape={x.shape}, adj.shape={adj.shape}")
print(f"输出: out.shape={out.shape}")
print(f"参数量: {sum(p.numel() for p in gat_layer.parameters()):,}")

# 展示注意力权重
with torch.no_grad():
    x_att = torch.relu(gat_layer.atoms_att(x))
    x_att_shape = adj.shape[:-1] + (x_att.shape[-1],)
    x_rows = x_att.unsqueeze(1).expand(x_att_shape)
    x_cols = x_att.unsqueeze(2).expand(x_att_shape)
    x_att_cat = torch.cat([x_rows, x_cols, adj], dim=-1)
    att_scores = gat_layer.final_att(x_att_cat)

    # 与 forward() 保持一致：先取单个注意力头，再沿邻居维归一化
    head_idx = 0
    att_scores_head = att_scores[:, :, :, head_idx]
    att_weights = torch.softmax(att_scores_head + soft_mask, dim=-1) * mask

print(f"\n注意力权重 (head {head_idx}):")
print(att_weights[0].numpy().round(3))
print("每行和:", att_weights[0].sum(dim=-1).numpy().round(3), "(应接近 1.0)")
print("\n✅ GAT 层验证完成")

Step 2: MultiHeadGraphConvLayer 演示

输入: x.shape=torch.Size([1, 3, 64]), adj.shape=torch.Size([1, 3, 3, 8])
输出: out.shape=torch.Size([1, 3, 64])
参数量: 6,824

注意力权重 (head 0):
[[0.31  0.367 0.323]
 [0.377 0.32  0.302]
 [0.447 0.287 0.267]]
每行和: [1. 1. 1.] (应接近 1.0)

✅ GAT 层验证完成


## Step 3: GAT 编码器 (MeganEncoder)

编码器负责将输入的分子图（嵌入后的节点和边特征）转换为高层节点表示。

### 架构特点
- **4 层 GAT 卷积** (`n_encoder_conv=4`)
- **残差连接 (Residual)**: 每 2 层添加一次残差，即第 1、3 层无残差，第 2、4 层有残差
- **在 Stateful 模式下**：编码器仅在第一步运行，后续步骤复用编码结果（通过 state 传递）

```
Input ──► GAT_1 ──► GAT_2 (+Residual) ──► GAT_3 ──► GAT_4 (+Residual) ──► Output
  │                    ↑                    │                  ↑
  └────────────────────┘                    └──────────────────┘
```

In [8]:
# ============================================================
# Step 3: 实现 MeganEncoder
# ============================================================

class MeganEncoder(nn.Module):
    """
    MEGAN GAT 编码器。
    
    在 stateful 模式下，仅在第一步执行（后续步骤复用 state）。
    包含 n_encoder_conv 个 GAT 层，每 2 层一次残差连接。
    """
    def __init__(self, hidden_dim, bond_emb_dim, n_encoder_conv=4,
                 enc_residual=True, enc_dropout=0.0):
        super().__init__()
        self.n_conv = n_encoder_conv
        self.residual = enc_residual
        self.dropout = nn.Dropout(enc_dropout) if enc_dropout > 0 else lambda x: x
        
        self.conv_layers = nn.ModuleList()
        for i in range(self.n_conv):
            conv = MultiHeadGraphConvLayer(
                bond_dim=bond_emb_dim, input_dim=hidden_dim,
                output_dim=hidden_dim, residual=False
            )
            self.conv_layers.append(conv)
    
    def forward(self, x):
        atom_feats = x['node_features']
        prev_atom_feats = atom_feats
        
        for i, conv in enumerate(self.conv_layers):
            residual = self.residual and i % 2 == 1  # 第 2、4 层加残差
            atom_feats = conv(atom_feats, x['adj'], x['conv_mask'], x['conv_soft_mask'],
                              apply_activation=not residual)
            atom_feats = self.dropout(atom_feats)
            if residual:
                atom_feats = torch.relu(atom_feats + prev_atom_feats)
                prev_atom_feats = atom_feats
        
        x['node_features'] = atom_feats
        return x


# === 演示 ===
print("=" * 60)
print("Step 3: MeganEncoder 演示")
print("=" * 60)

hidden_dim = 512
bond_emb_dim = 8

encoder = MeganEncoder(hidden_dim=hidden_dim, bond_emb_dim=bond_emb_dim, n_encoder_conv=4)

# 模拟输入
n_nodes = 10
x_enc = {
    'node_features': torch.randn(1, n_nodes, hidden_dim),
    'adj': torch.randn(1, n_nodes, n_nodes, bond_emb_dim),
    'conv_mask': torch.ones(1, n_nodes, n_nodes),
    'conv_soft_mask': torch.zeros(1, n_nodes, n_nodes),
}

with torch.no_grad():
    output = encoder(x_enc)

print(f"\n输入节点特征: {x_enc['node_features'].shape}")
print(f"输出节点特征: {output['node_features'].shape}")
print(f"编码器层数: {len(encoder.conv_layers)}")
print(f"  Layer 0: 无残差, 有 ReLU")
print(f"  Layer 1: 有残差 (+ prev) + ReLU")
print(f"  Layer 2: 无残差, 有 ReLU")
print(f"  Layer 3: 有残差 (+ prev) + ReLU")

n_params = sum(p.numel() for p in encoder.parameters())
print(f"\n编码器总参数量: {n_params:,}")
print("\n✅ MeganEncoder 验证完成")

Step 3: MeganEncoder 演示

输入节点特征: torch.Size([1, 10, 512])
输出节点特征: torch.Size([1, 10, 512])
编码器层数: 4
  Layer 0: 无残差, 有 ReLU
  Layer 1: 有残差 (+ prev) + ReLU
  Layer 2: 无残差, 有 ReLU
  Layer 3: 有残差 (+ prev) + ReLU

编码器总参数量: 1,186,336

✅ MeganEncoder 验证完成


## Step 4: GAT 解码器与动作预测头 (MeganDecoder)

解码器结构更为复杂，因为它需要同时产生两类输出：

### 原子动作头 (Atom Action Head)
```
node_features ──► FC_1 ──► FC_2 ──► FC_3 ──► (N, n_atom_actions)
```
预测每个节点上的原子级动作概率（如 change_atom、stop 等）

### 键动作头 (Bond Action Head)
```
node_features ──► FC_bond_atom ──► (N, bond_atom_dim)
                                         │
对每对节点 (i,j):  feat_i + feat_j + e_ij ──► FC_1 ──► FC_2 ──► FC_3 ──► (N, N, n_bond_actions)
```
预测每对节点之间的键级动作概率（如 change_bond、delete_bond、add_atom 等）

### 输出整合
原子动作和键动作被**展平并拼接**后，通过全局 softmax 归一化，确保所有可能动作的概率之和为 1。

In [9]:
# ============================================================
# Step 4: 实现 MeganDecoder
# ============================================================

class MeganDecoder(nn.Module):
    """
    MEGAN GAT 解码器，包含：
    1. n_decoder_conv 个 GAT 卷积层（带残差）
    2. 原子动作预测头 (FC layers → n_atom_actions)
    3. 键动作预测头 (FC layers → n_bond_actions)
    4. 全局 softmax 归一化
    """
    def __init__(self, hidden_dim, bond_emb_dim, n_atom_actions, n_bond_actions,
                 n_fc=3, n_decoder_conv=4, dec_residual=True,
                 bond_atom_dim=32, atom_fc_hidden_dim=256, bond_fc_hidden_dim=256,
                 dec_dropout=0.0):
        super().__init__()
        self.n_actions = n_atom_actions
        self.n_bond_actions = n_bond_actions
        self.hidden_dim = hidden_dim
        
        # GAT 卷积层
        self.conv_layers = nn.ModuleList()
        self.n_conv = n_decoder_conv
        self.residual = dec_residual
        self.dropout = nn.Dropout(dec_dropout) if dec_dropout > 0 else lambda x: x
        
        for i in range(n_decoder_conv):
            conv = MultiHeadGraphConvLayer(
                bond_dim=bond_emb_dim, input_dim=hidden_dim,
                output_dim=hidden_dim, residual=False
            )
            self.conv_layers.append(conv)
        
        # 原子动作头: hidden_dim → 256 → 256 → n_atom_actions
        self.fc_atom_layers = nn.ModuleList()
        for i in range(n_fc):
            in_dim = hidden_dim if i == 0 else atom_fc_hidden_dim
            out_dim = atom_fc_hidden_dim if i < n_fc - 1 else n_atom_actions
            self.fc_atom_layers.append(nn.Linear(in_dim, out_dim))
        
        # 键动作头: hidden_dim → bond_atom_dim, 然后 (2*bond_atom_dim + bond_emb) → 256 → 256 → n_bond_actions
        self.fc_atom_bond = nn.Linear(hidden_dim, bond_atom_dim)
        self.fc_bond_layers = nn.ModuleList()
        for i in range(n_fc):
            in_dim = bond_atom_dim + bond_emb_dim if i == 0 else bond_fc_hidden_dim
            out_dim = bond_fc_hidden_dim if i < n_fc - 1 else n_bond_actions
            self.fc_bond_layers.append(nn.Linear(in_dim, out_dim))
    
    def forward(self, x):
        atom_feats = x['node_features']
        prev_atom_feats = atom_feats
        
        # 1. GAT 卷积（与编码器类似，但独立参数）
        for i, conv in enumerate(self.conv_layers):
            residual = self.residual and i % 2 == 1
            atom_feats = conv(atom_feats, x['adj'], x['conv_mask'], x['conv_soft_mask'],
                              apply_activation=not residual)
            atom_feats = self.dropout(atom_feats)
            if residual:
                atom_feats = torch.relu(atom_feats + prev_atom_feats)
                prev_atom_feats = atom_feats
        
        # 掩码：只保留有效节点
        atom_feats = atom_feats * x['node_mask'].expand(*atom_feats.shape)
        node_state = atom_feats  # 保存状态用于 stateful 模式
        
        # 2. 原子动作头
        atom_out = atom_feats
        for layer in self.fc_atom_layers[:-1]:
            atom_out = torch.relu(layer(atom_out))
            atom_out = self.dropout(atom_out)
        atom_out = self.fc_atom_layers[-1](atom_out)  # (B, N, n_atom_actions)
        
        # 3. 键动作头
        bond_atom_feat = torch.relu(self.fc_atom_bond(atom_feats))  # (B, N, bond_atom_dim)
        # 对每对节点 (i,j): feat_i + feat_j
        exp_shape = x['adj'].shape[:-1] + (bond_atom_feat.shape[-1],)
        feat_rows = bond_atom_feat.unsqueeze(1).expand(exp_shape)
        feat_cols = bond_atom_feat.unsqueeze(2).expand(exp_shape)
        bond_feat = torch.cat([feat_rows + feat_cols, x['adj']], dim=-1)  # (B, N, N, bond_atom_dim+bond_emb)
        
        for layer in self.fc_bond_layers[:-1]:
            bond_feat = torch.relu(layer(bond_feat))
            bond_feat = self.dropout(bond_feat)
        bond_out = self.fc_bond_layers[-1](bond_feat)  # (B, N, N, n_bond_actions)
        
        # 4. 统一输出维度并拼接
        max_feat = max(atom_out.shape[-1], bond_out.shape[-1])
        B, N = atom_out.shape[0], atom_out.shape[1]
        
        atom_exp = torch.zeros(B, 1, N, max_feat, device=atom_feats.device)
        atom_exp[:, 0, :, :atom_out.shape[-1]] = atom_out
        
        bond_exp = torch.zeros(B, N, N, max_feat, device=atom_feats.device)
        bond_exp[:, :, :, :bond_out.shape[-1]] = bond_out
        
        # 拼接: (B, N+1, N, max_feat) → 展平 → softmax
        all_actions = torch.cat([bond_exp, atom_exp], dim=1)
        all_actions = all_actions.reshape(B, -1)
        
        # 创建掩码（简化版，实际会有更精细的 action_mask）
        all_mask = torch.ones_like(all_actions)
        soft_mask = torch.zeros_like(all_actions)
        
        output = torch.softmax(all_actions + soft_mask, dim=-1) * all_mask
        
        return node_state, output, all_mask


# === 演示 ===
print("=" * 60)
print("Step 4: MeganDecoder 演示")
print("=" * 60)

n_atom_actions = 159    # 原子动作数（MEGAN 默认）
n_bond_actions = 5      # 键动作数

decoder = MeganDecoder(
    hidden_dim=hidden_dim, bond_emb_dim=bond_emb_dim,
    n_atom_actions=n_atom_actions, n_bond_actions=n_bond_actions
)

# 模拟输入
n_nodes = 10
x_dec = {
    'node_features': torch.randn(1, n_nodes, hidden_dim),
    'adj': torch.randn(1, n_nodes, n_nodes, bond_emb_dim),
    'conv_mask': torch.ones(1, n_nodes, n_nodes),
    'conv_soft_mask': torch.zeros(1, n_nodes, n_nodes),
    'node_mask': torch.ones(1, n_nodes, 1),
}

with torch.no_grad():
    state, output, mask = decoder(x_dec)

print(f"\n输入节点特征: (1, {n_nodes}, {hidden_dim})")
print(f"输出 state: {state.shape}")
print(f"输出 output: {output.shape}")
print(f"  = (N+1) × N × max_actions = ({n_nodes}+1) × {n_nodes} × {max(n_atom_actions, n_bond_actions)}")
print(f"  = {(n_nodes + 1) * n_nodes * max(n_atom_actions, n_bond_actions)}")
print(f"输出概率之和: {output.sum().item():.4f} (应接近 1.0)")

n_params = sum(p.numel() for p in decoder.parameters())
print(f"\n解码器总参数量: {n_params:,}")
print("  - 4 层 GAT 卷积")
print(f"  - 原子动作头: 3 层 FC → {n_atom_actions} 个动作")
print(f"  - 键动作头: 3 层 FC → {n_bond_actions} 个动作")
print("\n✅ MeganDecoder 验证完成")

Step 4: MeganDecoder 演示

输入节点特征: (1, 10, 512)
输出 state: torch.Size([1, 10, 512])
输出 output: torch.Size([1, 17490])
  = (N+1) × N × max_actions = (10+1) × 10 × 159
  = 17490
输出概率之和: 1.0000 (应接近 1.0)

解码器总参数量: 1,518,308
  - 4 层 GAT 卷积
  - 原子动作头: 3 层 FC → 159 个动作
  - 键动作头: 3 层 FC → 5 个动作

✅ MeganDecoder 验证完成


## Step 5: MEGAN 完整模型

完整 MEGAN 模型整合了编码器和解码器，并负责：

1. **预处理 (`_preprocess`)**: 将整数特征转为 One-Hot → 线性嵌入
2. **Stateful 前向传播**: 
   - 第 1 步: 编码器处理 → 解码器预测
   - 第 2+ 步: 跳过编码器，将上一步 decoder 的 state 与新输入通过 `max()` 融合 → 解码器预测

### Stateful vs Stateless 模式
```
Stateful (默认):
  Step 1: Encoder → Decoder → state₁
  Step 2: max(embed(x₂), state₁) → Decoder → state₂
  Step 3: max(embed(x₃), state₂) → Decoder → state₃
  ...

Stateless:
  Step 1: Encoder → Decoder
  Step 2: Encoder → Decoder  (每步都重新编码)
  Step 3: Encoder → Decoder
```

Stateful 模式更高效（编码器只运行一次），同时通过 `max()` 操作保持对历史信息的记忆。

In [10]:
# ============================================================
# Step 5: 实现完整 MEGAN 模型
# ============================================================

def to_one_hot(x, dims):
    """将整数张量转换为 One-Hot 编码"""
    x_long = x.long()
    one_hot = torch.zeros(*x_long.shape, dims, device=x.device)
    one_hot.scatter_(x_long.dim(), x_long.unsqueeze(-1), 1)
    return one_hot


# MEGAN 使用的默认特征键
DEFAULT_ATOM_FEATURES = ('is_supernode', 'atomic_num', 'formal_charge', 'chiral_tag',
                         'num_explicit_hs', 'is_aromatic', 'is_edited')
DEFAULT_BOND_FEATURES = ('bond_type', 'bond_stereo', 'is_edited')


class MeganModel(nn.Module):
    """
    完整的 MEGAN 模型。
    
    前向传播流程:
    1. _preprocess: 整数特征 → One-Hot → 线性嵌入
    2. forward_step: 
       - 第 1 步: Encoder → Decoder
       - 后续步: max(embedding, state) → Decoder
    """
    def __init__(self, n_atom_actions, n_bond_actions, prop2oh,
                 bond_emb_dim=8, hidden_dim=512, stateful=True,
                 atom_feature_keys=DEFAULT_ATOM_FEATURES,
                 bond_feature_keys=DEFAULT_BOND_FEATURES):
        super().__init__()
        self.prop2oh = prop2oh
        self.hidden_dim = hidden_dim
        self.stateful = stateful
        
        # 计算 One-Hot 总维度
        total_atom_oh = sum(len(prop2oh['atom'].get(k, {})) + 1 for k in atom_feature_keys 
                            if k in prop2oh['atom'])
        total_bond_oh = sum(len(prop2oh['bond'].get(k, {})) + 1 for k in bond_feature_keys
                            if k in prop2oh['bond'])
        
        # 记录特征键和索引
        # ORDERED keys 是 mol_graph 中使用的所有可能特征键
        ORDERED_ATOM_KEYS = ['is_supernode', 'atomic_num', 'formal_charge', 'chiral_tag',
                             'num_explicit_hs', 'is_aromatic', 'is_edited', 'is_reactant']
        ORDERED_BOND_KEYS = ['bond_type', 'bond_stereo', 'is_edited']
        
        self.numbered_atom_oh_keys = [(ORDERED_ATOM_KEYS.index(k), k) for k in atom_feature_keys
                                       if k in ORDERED_ATOM_KEYS]
        self.numbered_bond_oh_keys = [(ORDERED_BOND_KEYS.index(k), k) for k in bond_feature_keys
                                       if k in ORDERED_BOND_KEYS]
        
        # 嵌入层
        self.atom_embedding = nn.Linear(total_atom_oh, hidden_dim)
        self.bond_embedding = nn.Linear(total_bond_oh, bond_emb_dim)
        
        # 编码器和解码器
        self.encoder = MeganEncoder(hidden_dim=hidden_dim, bond_emb_dim=bond_emb_dim)
        self.decoder = MeganDecoder(hidden_dim=hidden_dim, bond_emb_dim=bond_emb_dim,
                                     n_atom_actions=n_atom_actions, n_bond_actions=n_bond_actions)
    
    def _preprocess(self, x):
        """将整数特征转为 One-Hot 嵌入"""
        # 避免原地覆盖调用方输入，stateful 多步推理会复用离散特征 batch
        x = dict(x)

        # 原子特征 One-Hot
        oh_atom_feats = []
        for i, key in self.numbered_atom_oh_keys:
            oh_feat = to_one_hot(x['node_features'][:, :, i], dims=len(self.prop2oh['atom'][key]) + 1)
            oh_atom_feats.append(oh_feat)
        atom_feats = torch.cat(oh_atom_feats, dim=-1)
        x['node_features'] = self.atom_embedding(atom_feats)
        
        # 键特征 One-Hot
        oh_bond_feats = []
        for i, key in self.numbered_bond_oh_keys:
            oh_feat = to_one_hot(x['adj'][:, :, :, i], dims=len(self.prop2oh['bond'][key]) + 1)
            oh_bond_feats.append(oh_feat)
        adj = torch.cat(oh_bond_feats, dim=-1)
        x['adj'] = self.bond_embedding(adj)
        
        # 构建卷积掩码
        conv_mask = x['adj_mask'].float().squeeze(-1)
        x['conv_mask'] = conv_mask
        x['conv_soft_mask'] = (-conv_mask + 1.0) * -1e9
        
        # 键动作掩码
        node_adj_mask = x['node_mask'].unsqueeze(-1)
        node_adj_mask = node_adj_mask.expand(*node_adj_mask.shape)
        node_adj_mask = node_adj_mask * node_adj_mask.permute(0, 2, 1, 3).contiguous()
        x['node_adj_mask'] = node_adj_mask
        
        return x
    
    def forward_step(self, step_batch, state_dict=None):
        """
        单步前向传播。
        
        第 1 步 (state_dict=None): Encoder → Decoder
        后续步: max(embedding, state) → Decoder
        """
        step_batch = self._preprocess(step_batch)
        
        if self.stateful:
            if state_dict is None:
                # 第一步：完整编码
                step_batch = self.encoder(step_batch)
            else:
                # 后续步：融合 state
                state = state_dict['state']
                n_nodes = step_batch['node_features'].shape[1]
                if state.shape[1] != n_nodes:
                    min_n = min(state.shape[1], n_nodes)
                    new_state = torch.zeros(state.shape[0], n_nodes, self.hidden_dim, device=device)
                    new_state[:, :min_n] = state[:, :min_n]
                    state = new_state
                # element-wise max：既保留新嵌入信息，又保留历史 state
                step_batch['node_features'] = torch.max(step_batch['node_features'], state)
            
            state, output, mask = self.decoder(step_batch)
            return {'state': state, 'output': output, 'output_mask': mask}
        else:
            step_batch = self.encoder(step_batch)
            _, output, mask = self.decoder(step_batch)
            return {'output': output, 'output_mask': mask}


# === 演示 ===
print("=" * 60)
print("Step 5: 完整 MEGAN 模型演示")
print("=" * 60)

# 构建特征字典（简化版，用于演示）
demo_prop2oh = {
    'atom': {
        'is_supernode': {0: 1, 1: 2},
        'atomic_num': {6: 1, 7: 2, 8: 3, 9: 4, 16: 5, 17: 6, 35: 7, 53: 8},
        'formal_charge': {-1: 1, 0: 2, 1: 3},
        'chiral_tag': {0: 1, 1: 2, 2: 3},
        'num_explicit_hs': {0: 1, 1: 2, 2: 3, 3: 4},
        'is_aromatic': {0: 1, 1: 2},
        'is_edited': {0: 1, 1: 2},
    },
    'bond': {
        'bond_type': {1: 1, 2: 2, 3: 3, 12: 4, 'self': 5, 'supernode': 6},
        'bond_stereo': {0: 1, 1: 2, 6: 3},
        'is_edited': {0: 1, 1: 2},
    }
}

n_atom_actions = 159
n_bond_actions = 5

model = MeganModel(
    n_atom_actions=n_atom_actions,
    n_bond_actions=n_bond_actions,
    prop2oh=demo_prop2oh,
    hidden_dim=512,
    bond_emb_dim=8,
    stateful=True
)
model = model.to(device)
model.eval()

# 模拟 2 步自回归推理
n_nodes = 8
n_atom_feats = len(demo_prop2oh['atom'])  # 7 个原子特征
n_bond_feats = len(demo_prop2oh['bond'])  # 3 个键特征

fake_batch = {
    'node_features': torch.randint(0, 3, (1, n_nodes, n_atom_feats), device=device),
    'adj': torch.randint(0, 3, (1, n_nodes, n_nodes, n_bond_feats), device=device),
    'node_mask': torch.ones(1, n_nodes, 1, device=device),
    'adj_mask': torch.ones(1, n_nodes, n_nodes, 1, device=device),
}

with torch.no_grad():
    # Step 1: 编码器 + 解码器
    result1 = model.forward_step(fake_batch, state_dict=None)
    print(f"\nStep 1 (Encoder + Decoder):")
    print(f"  state: {result1['state'].shape}")
    print(f"  output: {result1['output'].shape}")
    print(f"  输出概率之和: {result1['output'].sum().item():.4f}")
    
    # Step 2: 仅解码器（复用 state）
    result2 = model.forward_step(fake_batch, state_dict={'state': result1['state']})
    print(f"\nStep 2 (State + Decoder, 跳过 Encoder):")
    print(f"  state: {result2['state'].shape}")
    print(f"  output: {result2['output'].shape}")
    print(f"  输出概率之和: {result2['output'].sum().item():.4f}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nMEGAN 模型总参数量: {total_params:,}")
print(f"  编码器: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"  解码器: {sum(p.numel() for p in model.decoder.parameters()):,}")
print(f"  嵌入层: {sum(p.numel() for p in model.atom_embedding.parameters()) + sum(p.numel() for p in model.bond_embedding.parameters()):,}")
print("\n✅ 完整 MEGAN 模型验证完成")

Step 5: 完整 MEGAN 模型演示

Step 1 (Encoder + Decoder):
  state: torch.Size([1, 8, 512])
  output: torch.Size([1, 11448])
  输出概率之和: 1.0000

Step 2 (State + Decoder, 跳过 Encoder):
  state: torch.Size([1, 8, 512])
  output: torch.Size([1, 11448])
  输出概率之和: 1.0000

MEGAN 模型总参数量: 2,721,148
  编码器: 1,186,336
  解码器: 1,518,308
  嵌入层: 16,504

✅ 完整 MEGAN 模型验证完成


## Step 6: 自回归推理与 Beam Search

### 自回归推理流程

MEGAN 的推理过程是**自回归 (Autoregressive)** 的：
1. 输入产物分子图
2. 模型预测最佳动作（从所有可能的原子/键动作中选择概率最高的）
3. 在分子上执行该动作
4. 将更新后的分子图作为输入，重新预测
5. 重复直到预测出 `Stop` 动作
6. 输出最终生成的反应物

### Beam Search
为了获得更好的结果，MEGAN 使用 **Beam Search**（束搜索）：
- 每步保留 top-k 个最可能的路径（而非仅保留最优）
- 最终从所有完成的路径中选择概率最高的

```
Step 1:  产物图 → [动作A (p=0.5), 动作B (p=0.3), 动作C (p=0.2)]
                         │               │               │
Step 2:              图A' →           图B' →           图C' →
                   [A1, A2, A3]     [B1, B2, B3]     [C1, C2, C3]
                         │               │
Step 3:          保留 top-k 路径继续搜索...
                         │
                      ...
                         │
Done:            选择概率最高的完整路径 → 反应物
```

### 动作解码
模型输出是一个展平的概率向量：
- `output.shape = (batch, (N+1) × N × max_actions)`
- 通过 `unravel_index` 还原为 `(atom1_type, atom1_ind, atom2_ind, action_ind)`
- 然后查表得到具体的动作元组

In [11]:
# ============================================================
# Step 6: 自回归推理与 Beam Search 伪代码演示
# ============================================================
# 注：由于没有训练好的模型权重，此处用伪代码和流程模拟展示

def greedy_decode(model, product_mol, max_steps=16):
    """
    贪心解码（Beam Size = 1）的伪代码。
    展示 MEGAN 自回归推理的核心逻辑。
    """
    mol = Chem.RWMol(product_mol)
    actions_taken = []
    state_dict = None
    
    for step in range(max_steps):
        # 1. 构建分子图
        # adj, nodes = get_graph(mol, ...)
        # batch = prepare_batch(adj, nodes)
        
        # 2. 模型前向传播
        # result = model.forward_step(batch, state_dict=state_dict)
        # state_dict = {'state': result['state']}
        
        # 3. 从输出概率中选择最佳动作
        # output = result['output']  # (1, n_total_actions)
        # best_action_ind = output.argmax()
        # action = decode_action(best_action_ind, n_nodes, action_vocab)
        
        # 4. 执行动作
        # mol = action.apply(mol)
        # actions_taken.append(action)
        
        # 5. 如果是 Stop，终止
        # if isinstance(action, StopAction):
        #     break
        pass
    
    # return Chem.MolToSmiles(mol), actions_taken
    return None, []


def beam_search_decode(model, product_mol, beam_size=10, max_steps=16):
    """
    Beam Search 解码的核心逻辑伪代码。
    """
    # 初始化：一条路径，起点为产物分子
    # paths = [{'mol': product_mol, 'prob': 1.0, 'actions': [], 'state': None}]
    # finished_paths = []
    
    # for step in range(max_steps):
    #     all_candidates = []
    #     
    #     # 对每条活跃路径
    #     for path in paths:
    #         batch = prepare_batch(path['mol'])
    #         result = model.forward_step(batch, state_dict=path['state'])
    #         
    #         # 取 top-k 个动作
    #         top_k_probs, top_k_actions = get_top_k(result['output'], beam_size)
    #         
    #         for prob, action in zip(top_k_probs, top_k_actions):
    #             new_mol = action.apply(path['mol'])
    #             new_path = {
    #                 'mol': new_mol,
    #                 'prob': path['prob'] * prob,
    #                 'actions': path['actions'] + [(action, prob)],
    #                 'state': result['state']
    #             }
    #             
    #             if isinstance(action, StopAction):
    #                 finished_paths.append(new_path)
    #             else:
    #                 all_candidates.append(new_path)
    #     
    #     # 保留 top beam_size 条路径
    #     paths = sorted(all_candidates, key=lambda p: -p['prob'])[:beam_size]
    # 
    # # 返回概率最高的完成路径
    # return sorted(finished_paths, key=lambda p: -p['prob'])
    pass


# === 推理流程可视化 ===
print("=" * 60)
print("Step 6: MEGAN 推理流程展示")
print("=" * 60)

print("""
╔══════════════════════════════════════════════════════════╗
║              MEGAN 自回归推理流程                        ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  输入: 产物 SMILES                                       ║
║    │                                                     ║
║    ▼                                                     ║
║  产物 → 分子图 (nodes, adj)                              ║
║    │                                                     ║
║    ▼                                                     ║
║  ┌──────────────────────────────────────┐                ║
║  │ Step 1: Encoder + Decoder            │                ║
║  │   整数特征 → One-Hot → 嵌入           │                ║
║  │   → 4层 GAT 编码                      │                ║
║  │   → 4层 GAT 解码                      │                ║
║  │   → 原子头 + 键头 → Softmax          │                ║
║  │   → 选择最佳动作 (或 top-k)           │                ║
║  │   → 保存 state                        │                ║
║  └──────────────┬───────────────────────┘                ║
║                 │                                        ║
║                 ▼                                        ║
║  ┌──────────────────────────────────────┐                ║
║  │ Step 2+: max(embed, state) → Decoder │                ║
║  │   （跳过 Encoder，复用 state）         │                ║
║  │   → 预测下一个动作                    │                ║
║  └──────────────┬───────────────────────┘                ║
║                 │                                        ║
║           ┌─────┴─────┐                                  ║
║           │ Stop?      │                                  ║
║           ├─ No  ──────┤──► 执行动作，更新分子图，回到↑    ║
║           └─ Yes ──────┘                                  ║
║                 │                                        ║
║                 ▼                                        ║
║  输出: 反应物 SMILES                                     ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""")

# 模拟一个完整推理路径的输出格式
print("示例推理输出（模拟）:")
print("-" * 50)
print(f"  产物: CC(=O)Oc1ccccc1C(=O)O")
print(f"  Step 0: DeleteBond(2-3)         [概率: 0.82]")
print(f"  Step 1: EditAtom(3): charge=-1  [概率: 0.91]")
print(f"  Step 2: AddAtom(O:15 to 2)      [概率: 0.76]")
print(f"  Step 3: EditAtom(15): hs=1      [概率: 0.88]")
print(f"  Step 4: Stop                    [概率: 0.95]")
print(f"  反应物: CC(=O)O.Oc1ccccc1C(=O)[O-]")
print()

# 动作空间大小说明
print("动作空间大小说明:")
print(f"  对于 N 个节点的分子图:")
print(f"  - 原子动作: N × {n_atom_actions} = N×{n_atom_actions}")
print(f"  - 键动作:   N × N × {n_bond_actions} = N²×{n_bond_actions}")
print(f"  - 总动作空间: N×{n_atom_actions} + N²×{n_bond_actions}")
print(f"  例如 N=20: 20×{n_atom_actions} + 400×{n_bond_actions} = {20*n_atom_actions + 400*n_bond_actions}")
print("\n✅ 推理流程展示完成")

Step 6: MEGAN 推理流程展示

╔══════════════════════════════════════════════════════════╗
║              MEGAN 自回归推理流程                        ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  输入: 产物 SMILES                                       ║
║    │                                                     ║
║    ▼                                                     ║
║  产物 → 分子图 (nodes, adj)                              ║
║    │                                                     ║
║    ▼                                                     ║
║  ┌──────────────────────────────────────┐                ║
║  │ Step 1: Encoder + Decoder            │                ║
║  │   整数特征 → One-Hot → 嵌入           │                ║
║  │   → 4层 GAT 编码                      │                ║
║  │   → 4层 GAT 解码                      │                ║
║  │   → 原子头 + 键头 → Softmax          │                ║
║  │   → 选择最佳动作 (或 top-k)       

## 总结

### MEGAN 模型架构回顾

| 组件 | 描述 | 参数 |
|------|------|------|
| `MultiHeadGraphConvLayer` | 基础 GAT 层，边特征参与注意力计算 | att_heads=8, att_dim=64 |
| `MeganEncoder` | 4 层 GAT + 残差连接 | hidden_dim=512 |
| `MeganDecoder` | 4 层 GAT + 原子/键动作预测头 | n_fc=3, hidden_dim=512 |
| `MeganModel` | 整合上述组件，支持 Stateful/Stateless | ~数百万参数 |

### MEGAN 的核心创新点

1. **图编辑序列**: 将化学反应建模为一系列图编辑操作，而非模板匹配或序列翻译
2. **动态图表示**: 分子图在每步动作后即时更新，模型基于最新状态做出下一个决策
3. **Stateful 机制**: 编码器只在第一步运行，后续步骤通过 `max()` 融合新特征与历史状态，既高效又保留长期记忆
4. **超节点**: 提供全局信息交换和 Stop 信号载体
5. **边感知注意力**: GAT 层同时考虑节点和边特征，更好地捕捉化学键信息

### 与其他方法的对比

| 方法类型 | 代表模型 | 优点 | 缺点 |
|---------|---------|------|------|
| 模板方法 | GLN, LocalRetro | 化学有效性高 | 受限于已知模板 |
| 序列方法 | G2Gs, Transformer | 灵活 | 可能生成无效分子 |
| **图编辑方法** | **MEGAN** | **既灵活又保证化学有效性** | **动作空间大** |

### 完整教程回顾
- `1_环境配置.ipynb`: 环境搭建与验证
- `2_数据处理.ipynb`: 分子图构建 → 特征编码 → 图编辑动作提取 → 稀疏矩阵存储
- `3_模型展示.ipynb`: GAT 编码器/解码器 → 完整模型 → 自回归推理与 Beam Search